# 01 - Data Preparation for Akkadian-to-English Translation

This notebook prepares training data for the Deep Past Challenge:
1. Extract sentence-level pairs from Sentences_Oare + published_texts
2. Build proper noun dictionary from onomasticon
3. Create augmented training dataset

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import defaultdict
import json
import warnings
warnings.filterwarnings('ignore')

# Paths - adjust for Kaggle vs local
try:
    # Kaggle paths
    BASE_PATH = Path('/kaggle/input/deep-past-initiative-machine-translation')
    ADDITIONAL_PATH = Path('/kaggle/input/old-assyrian-grammars-and-other-resources/archive')
    OUTPUT_PATH = Path('/kaggle/working')
except:
    # Local paths
    BASE_PATH = Path('data/deep-past-initiative-machine-translation')
    ADDITIONAL_PATH = Path('data/additional_data/archive')
    OUTPUT_PATH = Path('.')

print(f"Base path: {BASE_PATH}")
print(f"Additional path: {ADDITIONAL_PATH}")
print(f"Output path: {OUTPUT_PATH}")

## 1. Load Raw Data

In [ ]:
# Load primary training data
train_df = pd.read_csv(BASE_PATH / 'train.csv')
print(f"Training data: {len(train_df)} document-level pairs")
print(train_df.head(2))

In [ ]:
# Load sentence-level data (has translations but not full transliterations)
sentences_df = pd.read_csv(BASE_PATH / 'Sentences_Oare_FirstWord_LinNum.csv')
print(f"\nSentences data: {len(sentences_df)} sentence-level entries")
print(sentences_df.columns.tolist())
print(sentences_df.head(3))

In [ ]:
# Load published texts (has full transliterations)
published_df = pd.read_csv(BASE_PATH / 'published_texts.csv')
print(f"\nPublished texts: {len(published_df)} documents with transliterations")
print(published_df.columns.tolist())

In [ ]:
# Load onomasticon (proper names)
onomasticon_df = pd.read_csv(ADDITIONAL_PATH / 'onomasticon.csv')
print(f"\nOnomasticon: {len(onomasticon_df)} proper names")
print(onomasticon_df.head(3))

In [ ]:
# Load lexicon
lexicon_df = pd.read_csv(BASE_PATH / 'OA_Lexicon_eBL.csv')
print(f"\nLexicon: {len(lexicon_df)} entries")
print(f"Types: {lexicon_df['type'].value_counts().to_dict()}")

## 2. Build Proper Noun Dictionary

This dictionary maps Akkadian name spellings to their canonical English forms.

In [ ]:
def build_proper_noun_dict(onomasticon_df, lexicon_df):
    """
    Build a dictionary mapping Akkadian name spellings to canonical forms.
    """
    name_dict = {}
    
    # From onomasticon: spelling variants -> canonical name
    for _, row in onomasticon_df.iterrows():
        canonical = row['Name']
        spellings = str(row.get('Spellings_semicolon_separated', ''))
        
        if pd.notna(spellings) and spellings.strip():
            variants = [v.strip() for v in spellings.split(';') if v.strip()]
            for variant in variants:
                # Store lowercase for case-insensitive matching
                name_dict[variant.lower()] = canonical
        
        # Also add the canonical name itself
        if pd.notna(canonical):
            name_dict[canonical.lower()] = canonical
    
    # From lexicon: PN (Personal Name) entries
    pn_entries = lexicon_df[lexicon_df['type'] == 'PN']
    for _, row in pn_entries.iterrows():
        form = str(row.get('form', ''))
        norm = str(row.get('norm', ''))
        
        if pd.notna(form) and pd.notna(norm) and form.strip() and norm.strip():
            # Only add if not already in dict (onomasticon takes priority)
            if form.lower() not in name_dict:
                name_dict[form.lower()] = norm
    
    # From lexicon: GN (Geographic Name) entries
    gn_entries = lexicon_df[lexicon_df['type'] == 'GN']
    for _, row in gn_entries.iterrows():
        form = str(row.get('form', ''))
        norm = str(row.get('norm', ''))
        
        if pd.notna(form) and pd.notna(norm) and form.strip() and norm.strip():
            if form.lower() not in name_dict:
                name_dict[form.lower()] = norm
    
    return name_dict

# Build the dictionary
proper_noun_dict = build_proper_noun_dict(onomasticon_df, lexicon_df)
print(f"Built proper noun dictionary with {len(proper_noun_dict)} entries")

# Show some examples
print("\nSample entries:")
for i, (k, v) in enumerate(list(proper_noun_dict.items())[:10]):
    print(f"  '{k}' -> '{v}'")

In [ ]:
# Save the proper noun dictionary
with open(OUTPUT_PATH / 'proper_noun_dict.json', 'w', encoding='utf-8') as f:
    json.dump(proper_noun_dict, f, ensure_ascii=False, indent=2)
print(f"Saved proper noun dictionary to {OUTPUT_PATH / 'proper_noun_dict.json'}")

## 3. Extract Sentence-Level Training Pairs

The challenge: `Sentences_Oare` has translations but only `first_word_spelling`, not full transliterations.
We need to reconstruct sentence transliterations by matching with `published_texts.csv`.

In [ ]:
# Check overlap between sentences and published_texts
sentence_text_uuids = set(sentences_df['text_uuid'].unique())
published_oare_ids = set(published_df['oare_id'].unique())

overlap = sentence_text_uuids.intersection(published_oare_ids)
print(f"Unique text_uuids in sentences: {len(sentence_text_uuids)}")
print(f"Unique oare_ids in published: {len(published_oare_ids)}")
print(f"Overlap: {len(overlap)} documents")

In [ ]:
def parse_transliteration_lines(translit_text):
    """
    Parse a transliteration into lines based on common patterns.
    Returns a dict mapping line numbers to text content.
    """
    if pd.isna(translit_text) or not translit_text.strip():
        return {}
    
    lines = {}
    text = str(translit_text)
    
    # Split by common delimiters in transliterations
    # Akkadian texts often have words separated by spaces/hyphens
    # We'll treat the whole text as available content
    words = text.split()
    
    # For simplicity, create pseudo-lines based on word chunks
    # This is a heuristic - real line extraction would need tablet metadata
    chunk_size = 10  # words per "line"
    for i in range(0, len(words), chunk_size):
        line_num = (i // chunk_size) + 1
        lines[line_num] = ' '.join(words[i:i+chunk_size])
    
    return lines

In [ ]:
def extract_sentence_pairs(sentences_df, published_df, train_df):
    """
    Extract sentence-level transliteration-translation pairs.
    
    Strategy:
    1. For documents in train.csv, we have both transliteration and translation
    2. Use Sentences_Oare to identify sentence boundaries
    3. Split documents into sentences using first_word markers
    """
    sentence_pairs = []
    
    # Create lookup for train data
    train_lookup = {}
    for _, row in train_df.iterrows():
        train_lookup[row['oare_id']] = {
            'transliteration': row['transliteration'],
            'translation': row['translation']
        }
    
    # Create lookup for published texts
    published_lookup = {}
    for _, row in published_df.iterrows():
        published_lookup[row['oare_id']] = row['transliteration']
    
    # Group sentences by document
    doc_sentences = sentences_df.groupby('text_uuid')
    
    for text_uuid, doc_group in doc_sentences:
        # Check if we have this document's full transliteration
        if text_uuid not in published_lookup:
            continue
            
        full_translit = published_lookup[text_uuid]
        if pd.isna(full_translit) or not str(full_translit).strip():
            continue
        
        full_translit = str(full_translit)
        
        # Sort sentences by their position in the document
        doc_group = doc_group.sort_values('sentence_obj_in_text')
        
        # Extract each sentence
        sentences_list = doc_group.to_dict('records')
        
        for i, sent in enumerate(sentences_list):
            translation = sent.get('translation', '')
            first_word = sent.get('first_word_spelling', '')
            
            if pd.isna(translation) or not str(translation).strip():
                continue
            if pd.isna(first_word) or not str(first_word).strip():
                continue
                
            translation = str(translation).strip()
            first_word = str(first_word).strip()
            
            # Find the first word in the transliteration
            # Use word boundary matching
            pattern = re.escape(first_word)
            match = re.search(pattern, full_translit, re.IGNORECASE)
            
            if match:
                start_pos = match.start()
                
                # Find end position: start of next sentence or end of document
                if i + 1 < len(sentences_list):
                    next_first_word = sentences_list[i + 1].get('first_word_spelling', '')
                    if pd.notna(next_first_word) and str(next_first_word).strip():
                        next_pattern = re.escape(str(next_first_word).strip())
                        next_match = re.search(next_pattern, full_translit[start_pos + 1:], re.IGNORECASE)
                        if next_match:
                            end_pos = start_pos + 1 + next_match.start()
                        else:
                            end_pos = len(full_translit)
                    else:
                        end_pos = len(full_translit)
                else:
                    end_pos = len(full_translit)
                
                # Extract the sentence transliteration
                sentence_translit = full_translit[start_pos:end_pos].strip()
                
                # Only add if we have meaningful content
                if len(sentence_translit) > 5 and len(translation) > 3:
                    sentence_pairs.append({
                        'text_uuid': text_uuid,
                        'sentence_uuid': sent.get('sentence_uuid', ''),
                        'transliteration': sentence_translit,
                        'translation': translation
                    })
    
    return pd.DataFrame(sentence_pairs)

# Extract sentence pairs
sentence_pairs_df = extract_sentence_pairs(sentences_df, published_df, train_df)
print(f"Extracted {len(sentence_pairs_df)} sentence-level pairs")

In [ ]:
# Examine the extracted pairs
if len(sentence_pairs_df) > 0:
    print("\nSample extracted pairs:")
    for i in range(min(3, len(sentence_pairs_df))):
        row = sentence_pairs_df.iloc[i]
        print(f"\n--- Pair {i+1} ---")
        print(f"Translit: {row['transliteration'][:100]}...")
        print(f"Translation: {row['translation'][:100]}...")

## 4. Document-to-Sentence Splitting for train.csv

Split the original training documents into sentences using heuristics.

In [ ]:
def split_document_into_sentences(transliteration, translation):
    """
    Split a document-level pair into sentence-level pairs.
    
    Heuristics for splitting:
    1. Split translation on sentence-ending punctuation
    2. Proportionally split transliteration based on word counts
    """
    if pd.isna(transliteration) or pd.isna(translation):
        return []
    
    transliteration = str(transliteration).strip()
    translation = str(translation).strip()
    
    if not transliteration or not translation:
        return []
    
    # Split translation into sentences
    # Handle various sentence endings
    trans_sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', translation)
    trans_sentences = [s.strip() for s in trans_sentences if s.strip()]
    
    if len(trans_sentences) <= 1:
        # Can't split further, return as-is
        return [{'transliteration': transliteration, 'translation': translation}]
    
    # Split transliteration proportionally
    translit_words = transliteration.split()
    total_trans_words = sum(len(s.split()) for s in trans_sentences)
    
    pairs = []
    translit_idx = 0
    
    for trans_sent in trans_sentences:
        trans_word_count = len(trans_sent.split())
        # Estimate transliteration portion based on word ratio
        # Akkadian is typically denser, so we use a ratio
        ratio = trans_word_count / max(total_trans_words, 1)
        translit_word_count = int(len(translit_words) * ratio)
        translit_word_count = max(1, translit_word_count)  # At least 1 word
        
        # Extract transliteration portion
        end_idx = min(translit_idx + translit_word_count, len(translit_words))
        translit_portion = ' '.join(translit_words[translit_idx:end_idx])
        translit_idx = end_idx
        
        if translit_portion.strip() and trans_sent.strip():
            pairs.append({
                'transliteration': translit_portion.strip(),
                'translation': trans_sent.strip()
            })
    
    # Handle any remaining transliteration
    if translit_idx < len(translit_words) and pairs:
        remaining = ' '.join(translit_words[translit_idx:])
        pairs[-1]['transliteration'] += ' ' + remaining
    
    return pairs

# Split all training documents
all_sentence_pairs = []

for _, row in train_df.iterrows():
    pairs = split_document_into_sentences(row['transliteration'], row['translation'])
    for pair in pairs:
        pair['source'] = 'train_split'
        pair['oare_id'] = row['oare_id']
        all_sentence_pairs.append(pair)

train_split_df = pd.DataFrame(all_sentence_pairs)
print(f"Split training data into {len(train_split_df)} sentence pairs")

In [ ]:
# Show some examples of split pairs
print("\nSample split pairs:")
for i in range(min(3, len(train_split_df))):
    row = train_split_df.iloc[i]
    print(f"\n--- Split Pair {i+1} ---")
    print(f"Translit: {row['transliteration'][:80]}...")
    print(f"Translation: {row['translation'][:80]}...")

## 5. Combine All Training Data

In [ ]:
# Combine all sentence-level data
combined_data = []

# Add original document-level training data
for _, row in train_df.iterrows():
    combined_data.append({
        'transliteration': row['transliteration'],
        'translation': row['translation'],
        'source': 'original_document'
    })

# Add split sentence pairs
for _, row in train_split_df.iterrows():
    combined_data.append({
        'transliteration': row['transliteration'],
        'translation': row['translation'],
        'source': 'train_split'
    })

# Add extracted sentence pairs (if any)
if len(sentence_pairs_df) > 0:
    for _, row in sentence_pairs_df.iterrows():
        combined_data.append({
            'transliteration': row['transliteration'],
            'translation': row['translation'],
            'source': 'sentence_extraction'
        })

combined_df = pd.DataFrame(combined_data)
print(f"\nCombined training data: {len(combined_df)} pairs")
print(f"Source distribution:\n{combined_df['source'].value_counts()}")

In [ ]:
# Remove duplicates based on transliteration
combined_df_dedup = combined_df.drop_duplicates(subset=['transliteration'], keep='first')
print(f"After deduplication: {len(combined_df_dedup)} pairs")

In [ ]:
# Data quality checks
print("\nData quality statistics:")
print(f"Transliteration length - mean: {combined_df_dedup['transliteration'].str.len().mean():.1f}, "
      f"median: {combined_df_dedup['transliteration'].str.len().median():.1f}")
print(f"Translation length - mean: {combined_df_dedup['translation'].str.len().mean():.1f}, "
      f"median: {combined_df_dedup['translation'].str.len().median():.1f}")

# Filter out very short or very long pairs
min_translit_len = 10
max_translit_len = 2000
min_trans_len = 5
max_trans_len = 5000

filtered_df = combined_df_dedup[
    (combined_df_dedup['transliteration'].str.len() >= min_translit_len) &
    (combined_df_dedup['transliteration'].str.len() <= max_translit_len) &
    (combined_df_dedup['translation'].str.len() >= min_trans_len) &
    (combined_df_dedup['translation'].str.len() <= max_trans_len)
]

print(f"\nAfter length filtering: {len(filtered_df)} pairs")

In [ ]:
# Save the combined training data
filtered_df.to_csv(OUTPUT_PATH / 'train_augmented.csv', index=False)
print(f"Saved augmented training data to {OUTPUT_PATH / 'train_augmented.csv'}")

## 6. Create Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split by source to ensure balanced representation
train_data, val_data = train_test_split(
    filtered_df, 
    test_size=0.1, 
    random_state=42,
    stratify=filtered_df['source']
)

print(f"Training set: {len(train_data)} pairs")
print(f"Validation set: {len(val_data)} pairs")

# Save splits
train_data.to_csv(OUTPUT_PATH / 'train_final.csv', index=False)
val_data.to_csv(OUTPUT_PATH / 'val_final.csv', index=False)

print(f"\nSaved train_final.csv and val_final.csv")

## 7. Summary

In [ ]:
print("="*60)
print("DATA PREPARATION SUMMARY")
print("="*60)
print(f"\nOriginal training data: {len(train_df)} document pairs")
print(f"Sentence-level extracted: {len(sentence_pairs_df)} pairs")
print(f"Document splits: {len(train_split_df)} pairs")
print(f"Combined (deduplicated): {len(filtered_df)} pairs")
print(f"\nFinal training set: {len(train_data)} pairs")
print(f"Final validation set: {len(val_data)} pairs")
print(f"\nProper noun dictionary: {len(proper_noun_dict)} entries")
print("\nOutput files:")
print(f"  - {OUTPUT_PATH / 'train_augmented.csv'}")
print(f"  - {OUTPUT_PATH / 'train_final.csv'}")
print(f"  - {OUTPUT_PATH / 'val_final.csv'}")
print(f"  - {OUTPUT_PATH / 'proper_noun_dict.json'}")
print("="*60)